# 부서 연결 Retriever v4.1

## v3 → v4 주요 변경사항

| 구분 | 이슈 | 변경 내용 |
|------|------|----------|
| **[C-1] Critical** | GPU 강제 종료 코드 | `TRAIN_MODE` 플래그 분리 → 추론 전용 환경(CPU)에서도 기동 가능 |
| **[C-2] Critical** | 128토큰 Truncation | `preprocess_query_for_matching()` 추가 + `max_seq_length=256` 확장 |
| **[C-3] Critical** | Stage2 모델 로드 검증 | `verify_model_dir()` 필수 파일 체크 + 단계별 fallback |
| **[H-1] High** | Train-Serve Skew | Stage2 학습 시 다중 업무 조합 augmentation 추가 |
| **[H-2] High** | 다수결 5행 한계 | `REFERENCE_TOP_ROWS` 5→15, `CANDIDATE_K` 80→120 |
| **[H-3] High** | confidence_score 없음 | 득표율·점수집중도·갭 기반 0~1 신뢰도 수치 반환 |

> **max_position_embeddings=514** 확인: 아키텍처 상한 내에서 추론 시 256토큰까지 확장 가능


In [ ]:
# Colab Google Drive 마운트 (로컬 환경에서는 건너뜁니다)
import sys

IN_COLAB = "google.colab" in sys.modules
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Colab 환경 감지 — Drive 마운트 완료")
except ImportError:
    IN_COLAB = False
    print("로컬 환경 — Drive 마운트 생략")


In [ ]:
# 최초 1회 설치
# GPU 메모리 부족 시 STAGE1_BATCH_SIZE / STAGE2_BATCH_SIZE 를 줄이세요.
!pip install -q -U sentence-transformers rank_bm25 python-docx openpyxl tqdm scikit-learn datasets
print("패키지 설치 완료")


In [ ]:
# ============================================================
# 1. 라이브러리 및 기본 설정
# ============================================================
import os, re, pickle, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

# TRAIN_MODE = True  : 학습 실행 (GPU 권장, BGE-M3 570M params)
# TRAIN_MODE = False : 기존 모델 로드 후 추론만 실행
TRAIN_MODE = True

if TRAIN_MODE and DEVICE != "cuda":
    print("WARNING: GPU 없음. BGE-M3(570M params) 학습은 매우 느립니다.")
    print("  추론만 실행하려면 TRAIN_MODE = False 로 변경하세요.")
elif DEVICE == "cuda":
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    except Exception:
        pass
    print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

print(f"DEVICE: {DEVICE} | USE_AMP: {USE_AMP} | TRAIN_MODE: {TRAIN_MODE}")


In [ ]:
# ============================================================
# 2. 파일 경로 및 학습 옵션
# ============================================================
from pathlib import Path

PROJECT_DIR = Path('.')

BASE_DOC_PATH    = str(PROJECT_DIR / "업무담당규정.docx")
DEPT_XLSX_PATH   = str(PROJECT_DIR / "dept.xlsx")
# 단일 최종 모델 경로 — Stage1 저장 후 Stage2가 덮어씀
FINETUNED_MODEL_DIR = str(PROJECT_DIR / "bge_m3_finetuned")
STAGE1_MODEL_DIR    = FINETUNED_MODEL_DIR   # Stage1 저장 경로
STAGE2_MODEL_DIR    = FINETUNED_MODEL_DIR   # Stage2 덮어쓰기 경로
INDEX_CACHE_DIR  = str(PROJECT_DIR / "index_cache")

# ── base 모델 ──────────────────────────────────────────────
# BAAI/bge-m3: 570M params, max_seq_length=8192, 한국어 포함 100+ 언어
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"

# ── 학습 제어 ──────────────────────────────────────────────
RUN_STAGE1_DOMAIN_TRAIN = True
RUN_STAGE2_DEPT_TRAIN   = True

# ── Stage 1 ── (도메인 적응, BGE-M3는 이미 강력 → 1 epoch으로 충분)
STAGE1_EPOCHS          = 1
STAGE1_BATCH_SIZE      = 8    # OOM 시 4로 줄이기 (A100 기준 8 권장)
STAGE1_LR              = 1e-5  # BGE-M3 대형 모델: 2e-5 → 1e-5
STAGE1_MAX_TRAIN_TEXTS = 3000  # docx 문단 최대 수

# ── Stage 2 ── (부서 매칭 파인튜닝)
STAGE2_EPOCHS     = 2          # 데이터 적으므로 2 epoch
STAGE2_BATCH_SIZE = 8          # OOM 시 4로 줄이기
STAGE2_LR         = 1e-5

# ── Chunk ──────────────────────────────────────────────────
DEPT_CHUNK_SIZE = 5

# ── 검색 가중치 ────────────────────────────────────────────
DENSE_WEIGHT  = 0.55
TFIDF_WEIGHT  = 0.25
BM25_WEIGHT   = 0.20

ENCODE_BATCH_SIZE = 32   # BGE-M3: 인코딩 배치 (메모리 여유 시 64까지 가능)
CANDIDATE_K       = 120

# Google Drive 저장 경로 (Colab에서 Drive 마운트 시 자동 사용, 아니면 로컬 저장)
DRIVE_BASE      = '/content/drive/MyDrive'
DRIVE_FINETUNED = DRIVE_BASE + '/bge_m3_finetuned'
DRIVE_STAGE1    = DRIVE_FINETUNED   # Stage1 = Stage2 같은 경로
DRIVE_STAGE2    = DRIVE_FINETUNED

# BGE-M3 max_seq_length = 8192 (기본값) — 별도 설정 불필요
print(f"base 모델      : {EMBEDDING_MODEL_NAME}")
print(f"CANDIDATE_K    : {CANDIDATE_K}")
print(f"max_seq_length : 8192 (BGE-M3 기본값)")


In [ ]:
# ============================================================
# 2-1. 파일 존재 확인
# ============================================================
check_paths = {}
if TRAIN_MODE:
    check_paths = {
        "업무담당규정.docx": BASE_DOC_PATH,
        "dept.xlsx":         DEPT_XLSX_PATH,
    }

for name, p in check_paths.items():
    if not Path(p).exists():
        raise FileNotFoundError(
            f"[{name}] 파일을 찾을 수 없습니다: {p}\n"
            f"PROJECT_DIR={PROJECT_DIR.resolve()} 경로를 확인하세요."
        )

print("파일 확인 완료")


In [ ]:
# ============================================================
# 3. dept.xlsx 로드 및 검증
# ============================================================
def clean_text(x) -> str:
    x = "" if pd.isna(x) else str(x)
    return re.sub(r"\s+", " ", x).strip()


def load_dept_excel(path: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"dept.xlsx 없음: {path.resolve()}")

    df = pd.read_excel(path)
    required_cols = ["HNG_BR_NM", "OFWRK_NM"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"필수 컬럼 없음: {missing}  현재 컬럼: {list(df.columns)}")

    df = df[required_cols].copy()
    df["HNG_BR_NM"] = df["HNG_BR_NM"].apply(clean_text)
    df["OFWRK_NM"]  = df["OFWRK_NM"].apply(clean_text)
    df = df.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    df = df.dropna(subset=["HNG_BR_NM", "OFWRK_NM"]).reset_index(drop=True)
    df["orig_row_id"] = np.arange(len(df))
    return df


dept_df = load_dept_excel(DEPT_XLSX_PATH)
print("dept_df shape:", dept_df.shape)
print("부서 수:", dept_df["HNG_BR_NM"].nunique())
display(dept_df.head())


In [ ]:
# ============================================================
# 4. 업무담당규정.docx 로드 (1차 도메인 적응 학습용)
# ============================================================
def read_docx_paragraphs(docx_path: str, min_chars: int = 20) -> list:
    from docx import Document
    path = Path(docx_path)
    if not path.exists():
        raise FileNotFoundError(f"업무담당규정.docx 없음: {path.resolve()}")

    doc = Document(str(path))
    paragraphs = []
    for p in doc.paragraphs:
        t = clean_text(p.text)
        if len(t) >= min_chars:
            paragraphs.append(t)
    for table in doc.tables:
        for row in table.rows:
            merged = " ".join(clean_text(c.text) for c in row.cells if clean_text(c.text))
            if len(merged) >= min_chars:
                paragraphs.append(merged)

    seen, unique = set(), []
    for t in paragraphs:
        if t not in seen:
            unique.append(t); seen.add(t)
    return unique


domain_texts = read_docx_paragraphs(BASE_DOC_PATH, min_chars=20)
print("도메인 학습 문단 수:", len(domain_texts))
for t in domain_texts[:3]:
    print(" -", t[:100])


In [ ]:
# ============================================================
# [BGE-M3] 공시 본문 전처리 함수 — 길이 제한 없음
# ============================================================
# BGE-M3 max_seq_length=8192 → 금융 공시 본문 전체 입력 가능
# 제목 + 담당부서 힌트 + 본문 전체를 구조화하여 전달

_DEPT_HINT_RE = re.compile(
    r'담당\s*(부서|과|팀|실|국|원|관|처)',
    re.UNICODE
)

def preprocess_query_for_matching(text: str) -> str:
    if not text or not text.strip():
        return ""

    lines = [l.strip() for l in text.split("\n") if l.strip()]

    # 제목: 첫 번째 비공백 줄
    title = lines[0] if lines else ""

    # 담당부서 힌트 줄 (최대 3줄)
    dept_hints = []
    for line in lines[1:]:
        if _DEPT_HINT_RE.search(line):
            dept_hints.append(line)
        if len(dept_hints) >= 3:
            break

    # 본문 전체 (길이 제한 없음 — BGE-M3가 8192 tokens까지 처리)
    body_lines = [l for l in lines[1:] if l not in dept_hints]
    body = " ".join(body_lines)

    return " ".join(p for p in [title] + dept_hints + [body] if p)


# ── 전처리 효과 확인 ─────────────────────────────────────────
for fname in ["query1.txt", "query2.txt", "query3.txt"]:
    fpath = PROJECT_DIR / fname
    if fpath.exists():
        raw = fpath.read_text(encoding="utf-8")
        processed = preprocess_query_for_matching(raw)
        print(f"[{fname}] 원본 {len(raw)}자 → 전처리 후 {len(processed)}자")
        print(f"  내용: {processed[:120]}...")
        print()


In [ ]:
# ============================================================
# 5. Stage 1 학습: 도메인 적응 (업무담당규정.docx)
# ============================================================
# SentenceTransformerTrainer API (sentence-transformers >= 3.0)
# 자기 지도 방식: (문단, 동일 문단) 쌍 → 금융/내부업무 도메인 어휘 적응

def prepare_stage1_texts(domain_texts, max_train_texts=3000):
    texts = [clean_text(t) for t in domain_texts if len(clean_text(t)) >= 20]
    texts = list(dict.fromkeys(texts))
    if max_train_texts and len(texts) > max_train_texts:
        random.shuffle(texts); texts = texts[:max_train_texts]
    return texts


def train_stage1_domain_model(base_model_name, domain_texts, output_dir,
                               epochs=1, batch_size=8, lr=1e-5, max_train_texts=3000):
    texts = prepare_stage1_texts(domain_texts, max_train_texts)
    if not texts:
        raise ValueError("1차 학습용 domain_texts가 비어 있습니다.")
    print(f"1차 학습 예제 수: {len(texts)}")

    # anchor = positive = 동일 문단 (자기 지도 도메인 적응)
    train_ds = Dataset.from_dict({"anchor": texts, "positive": texts})

    model   = SentenceTransformer(base_model_name, device=DEVICE)
    loss_fn = MultipleNegativesRankingLoss(model)

    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=lr,
        warmup_ratio=0.1,
        fp16=USE_AMP,
        save_strategy="no",
        logging_steps=50,
        dataloader_drop_last=True,
    )
    trainer = SentenceTransformerTrainer(
        model=model, args=args, train_dataset=train_ds, loss=loss_fn,
    )
    trainer.train()

    # GPU 메모리 정리 (OOM 방지)
    import gc; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

    # Google Drive 우선 저장 (세션 종료 후에도 파일 보존)
    from pathlib import Path as _P
    if IN_COLAB and _P(DRIVE_BASE).parent.exists():
        _P(DRIVE_STAGE1).mkdir(parents=True, exist_ok=True)
        save_path = DRIVE_STAGE1
        print("Google Drive 저장:", save_path)
    else:
        _P(output_dir).mkdir(parents=True, exist_ok=True)
        save_path = output_dir

    model.save_pretrained(save_path)

    # 저장 검증
    wf = list(_P(save_path).glob("model.safetensors")) + list(_P(save_path).glob("pytorch_model.bin"))
    if wf:
        size_gb = round(wf[0].stat().st_size / 1e9, 2)
        print("Stage1 저장 완료:", save_path, str(size_gb) + " GB")
    else:
        print("WARNING: 가중치 파일 없음 —", save_path)
    return model


if TRAIN_MODE and RUN_STAGE1_DOMAIN_TRAIN:
    stage1_model = train_stage1_domain_model(
        base_model_name=EMBEDDING_MODEL_NAME,
        domain_texts=domain_texts,
        output_dir=STAGE1_MODEL_DIR,
        epochs=STAGE1_EPOCHS,
        batch_size=STAGE1_BATCH_SIZE,
        lr=STAGE1_LR,
        max_train_texts=STAGE1_MAX_TRAIN_TEXTS,
    )
    STAGE1_BASE_FOR_STAGE2 = STAGE1_MODEL_DIR
else:
    STAGE1_BASE_FOR_STAGE2 = (
        STAGE1_MODEL_DIR if Path(STAGE1_MODEL_DIR).exists()
        else EMBEDDING_MODEL_NAME
    )
    print("Stage1 학습 생략. 베이스:", STAGE1_BASE_FOR_STAGE2)


In [ ]:
# ============================================================
# 6. dept.xlsx 부서별 5행 chunk 생성
# ============================================================
def build_dept_chunks_by_department(dept_df, chunk_size=5):
    rows, chunk_no = [], 0
    for dept, g in dept_df.groupby("HNG_BR_NM", sort=False):
        g = g.sort_values("orig_row_id").reset_index(drop=True)
        for start in range(0, len(g), chunk_size):
            part  = g.iloc[start:start + chunk_size]
            works = part["OFWRK_NM"].astype(str).tolist()
            text  = (f"[부서명] {dept}\n[담당업무]\n" +
                     "\n".join(f"- {w}" for w in works))
            rows.append({
                "source"        : "dept.xlsx",
                "chunk_id"      : f"dept_chunk_{chunk_no}",
                "HNG_BR_NM"     : dept,
                "text"          : text,
                "work_items"    : works,
                "n_rows"        : len(part),
                "orig_row_start": int(part["orig_row_id"].min()),
                "orig_row_end"  : int(part["orig_row_id"].max()),
            })
            chunk_no += 1
    return pd.DataFrame(rows)


corpus_df = build_dept_chunks_by_department(dept_df, chunk_size=DEPT_CHUNK_SIZE)
print("corpus_df shape:", corpus_df.shape)
print("부서 수:", corpus_df["HNG_BR_NM"].nunique())
display(corpus_df.head(5))


In [ ]:
# ============================================================
# 7. Stage 2 학습: 부서 연결 파인튜닝
# ============================================================
# SentenceTransformerTrainer API
# 입력 쌍 3종류:
#   ① 단일 업무 문장 → 해당 부서 chunk
#   ② "부서명 + 업무" → 해당 부서 chunk
#   ③ 동일 부서 3~5개 업무 조합(랜덤) → 해당 부서 chunk  ← augmentation

def prepare_stage2_dept_examples(corpus_df, augment_multi=True):
    examples = []

    for _, row in corpus_df.iterrows():
        dept       = str(row["HNG_BR_NM"])
        chunk_text = str(row["text"])
        work_items = row["work_items"]

        for w in work_items:
            w = clean_text(w)
            if len(w) >= 3:
                examples.append(InputExample(texts=[w, chunk_text]))
                examples.append(InputExample(texts=[f"{dept} {w}", chunk_text]))
        examples.append(InputExample(texts=[dept, chunk_text]))

    if augment_multi:
        for dept, group in corpus_df.groupby("HNG_BR_NM"):
            all_works = []
            for _, row in group.iterrows():
                all_works.extend(row["work_items"])
            all_works = [clean_text(w) for w in all_works if len(clean_text(w)) >= 3]
            if len(all_works) < 2:
                continue

            first_chunk = str(group.iloc[0]["text"])
            full_q = f"[{dept}] " + " ".join(all_works)
            examples.append(InputExample(texts=[full_q[:600], first_chunk]))

            for _ in range(2):
                n = min(random.randint(3, 5), len(all_works))
                sampled_q = " ".join(random.sample(all_works, n))
                examples.append(InputExample(texts=[sampled_q, first_chunk]))

    random.shuffle(examples)
    return examples


def train_stage2_dept_model(base_model_name, corpus_df, output_dir,
                             epochs=2, batch_size=8, lr=1e-5):
    examples = prepare_stage2_dept_examples(corpus_df, augment_multi=True)
    if not examples:
        raise ValueError("2차 학습용 dept 예제가 비어 있습니다.")
    print(f"2차 학습 예제 수: {len(examples)}  (augmentation 포함)")

    train_ds = Dataset.from_dict({
        "anchor":   [ex.texts[0] for ex in examples],
        "positive": [ex.texts[1] for ex in examples],
    })

    model   = SentenceTransformer(base_model_name, device=DEVICE)
    loss_fn = MultipleNegativesRankingLoss(model)

    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=lr,
        warmup_ratio=0.1,
        fp16=USE_AMP,
        save_strategy="no",
        logging_steps=50,
        dataloader_drop_last=True,
    )
    trainer = SentenceTransformerTrainer(
        model=model, args=args, train_dataset=train_ds, loss=loss_fn,
    )
    trainer.train()

    # GPU 메모리 정리 (OOM 방지)
    import gc; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

    # Google Drive 우선 저장 (세션 종료 후에도 파일 보존)
    from pathlib import Path as _P
    if IN_COLAB and _P(DRIVE_BASE).parent.exists():
        _P(DRIVE_STAGE2).mkdir(parents=True, exist_ok=True)
        save_path = DRIVE_STAGE2
        print("Google Drive 저장:", save_path)
    else:
        _P(output_dir).mkdir(parents=True, exist_ok=True)
        save_path = output_dir

    model.save_pretrained(save_path)

    # 저장 검증
    wf = list(_P(save_path).glob("model.safetensors")) + list(_P(save_path).glob("pytorch_model.bin"))
    if wf:
        size_gb = round(wf[0].stat().st_size / 1e9, 2)
        print("Stage2 저장 완료:", save_path, str(size_gb) + " GB")
    else:
        print("WARNING: 가중치 파일 없음 —", save_path)
    return model


if TRAIN_MODE and RUN_STAGE2_DEPT_TRAIN:
    embedder = train_stage2_dept_model(
        base_model_name=STAGE1_BASE_FOR_STAGE2,
        corpus_df=corpus_df,
        output_dir=STAGE2_MODEL_DIR,
        epochs=STAGE2_EPOCHS,
        batch_size=STAGE2_BATCH_SIZE,
        lr=STAGE2_LR,
    )
else:
    print("Stage2 학습 생략 (모델 로드는 다음 셀에서)")


In [ ]:
# ============================================================
# [BGE-M3] 모델 로드 — 필수 파일 검증 + fallback
# ============================================================
# max_seq_length 별도 설정 불필요 — BGE-M3 기본값 8192

def verify_model_dir(model_dir: str) -> bool:
    p = Path(model_dir)
    if not p.exists():
        return False
    required = ["config.json", "tokenizer_config.json"]
    model_weights = ["model.safetensors", "pytorch_model.bin"]
    for f in required:
        if not (p / f).exists():
            return False
    if not any((p / mf).exists() for mf in model_weights):
        return False
    return True


if not (TRAIN_MODE and RUN_STAGE2_DEPT_TRAIN):
    if verify_model_dir(FINETUNED_MODEL_DIR):
        print("[OK] BGE-M3 파인튜닝 모델 로드:", FINETUNED_MODEL_DIR)
        embedder = SentenceTransformer(FINETUNED_MODEL_DIR, device=DEVICE)
    else:
        print("[WARN] 파인튜닝 모델 없음 → BGE-M3 기본 사용:", EMBEDDING_MODEL_NAME)
        embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

# BGE-M3 기본 max_seq_length = 8192 — 그대로 사용
print(f"[OK] 임베더 준비: max_seq_length={embedder.max_seq_length}")


In [ ]:
# ============================================================
# 8. 임베딩 / TF-IDF / BM25 인덱스 생성 + pickle 캐시
# ============================================================
# 서버 재시작 시 재인코딩 방지: numpy/pickle 저장

Path(INDEX_CACHE_DIR).mkdir(exist_ok=True)
_EMBS_F   = Path(INDEX_CACHE_DIR) / "corpus_embs.npy"
_TFIDF_F  = Path(INDEX_CACHE_DIR) / "tfidf.pkl"
_BM25_F   = Path(INDEX_CACHE_DIR) / "bm25.pkl"
_CORP_F   = Path(INDEX_CACHE_DIR) / "corpus_df.pkl"

corpus_texts = corpus_df["text"].astype(str).tolist()


def simple_tokenize(text):
    return re.findall(r"[가-힣A-Za-z0-9]+", str(text).lower())

def normalize_matrix(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def top_indices(scores, k):
    k = min(k, len(scores))
    return np.argsort(scores)[-k:][::-1]


# 학습 직후이면 캐시 강제 갱신
_force_rebuild = TRAIN_MODE and (RUN_STAGE1_DOMAIN_TRAIN or RUN_STAGE2_DEPT_TRAIN)

if (not _force_rebuild
        and _EMBS_F.exists() and _TFIDF_F.exists()
        and _BM25_F.exists() and _CORP_F.exists()):
    print("캐시에서 인덱스 로드...")
    corpus_embs = np.load(str(_EMBS_F))
    with open(_TFIDF_F, "rb") as f:
        tfidf_vectorizer, tfidf_matrix = pickle.load(f)
    with open(_BM25_F, "rb") as f:
        bm25 = pickle.load(f)
    corpus_df = pd.read_pickle(str(_CORP_F))
    print(f"  corpus_embs: {corpus_embs.shape}")
else:
    print("인덱스 새로 생성 중...")
    corpus_embs = normalize_matrix(
        embedder.encode(corpus_texts, convert_to_numpy=True,
                        show_progress_bar=True, batch_size=ENCODE_BATCH_SIZE)
    )
    tfidf_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=1)
    tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_texts)
    tokenized_corpus = [simple_tokenize(t) for t in corpus_texts]
    bm25 = BM25Okapi(tokenized_corpus)

    np.save(str(_EMBS_F), corpus_embs)
    with open(_TFIDF_F, "wb") as f:
        pickle.dump((tfidf_vectorizer, tfidf_matrix), f)
    with open(_BM25_F, "wb") as f:
        pickle.dump(bm25, f)
    corpus_df.to_pickle(str(_CORP_F))
    print(f"  corpus_embs: {corpus_embs.shape} | 캐시 저장 완료")


In [ ]:
# ============================================================
# 9. 검색 + 예측 함수 — top-1 직접 출력
# ============================================================

def minmax_norm(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return x
    return (x - x.min()) / (np.ptp(x) + 1e-12)


def retrieve_scored_rows(query, candidate_k=CANDIDATE_K, top_k=10,
                          dense_weight=DENSE_WEIGHT,
                          tfidf_weight=TFIDF_WEIGHT,
                          bm25_weight=BM25_WEIGHT):
    query = clean_text(query)
    if not query:
        raise ValueError("query가 비어 있습니다.")

    candidate_k = min(candidate_k, len(corpus_df))
    top_k       = min(top_k, len(corpus_df))

    q_emb = embedder.encode([query], convert_to_numpy=True)
    q_emb /= (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-12)
    dense_all  = corpus_embs @ q_emb[0]
    tfidf_all  = (tfidf_matrix @ tfidf_vectorizer.transform([query]).T).toarray().ravel()
    bm25_all   = np.asarray(bm25.get_scores(simple_tokenize(query)))

    cand = np.unique(np.concatenate([
        top_indices(dense_all, candidate_k),
        top_indices(tfidf_all, candidate_k),
        top_indices(bm25_all,  candidate_k),
    ]))

    d, t, b = dense_all[cand], tfidf_all[cand], bm25_all[cand]
    scores  = (dense_weight  * minmax_norm(d) +
               tfidf_weight  * minmax_norm(t) +
               bm25_weight   * minmax_norm(b))

    result = corpus_df.iloc[cand].copy()
    result["dense_score"] = d
    result["tfidf_score"] = t
    result["bm25_score"]  = b
    result["score"]       = scores
    result = result.sort_values("score", ascending=False).reset_index(drop=True)
    result = result.head(top_k).copy()
    result["rank"] = np.arange(1, len(result) + 1)
    return result


def predict_department(query, candidate_k=CANDIDATE_K, preprocess=True):
    raw_query   = query
    query_input = preprocess_query_for_matching(query) if preprocess else clean_text(query)

    scored_rows = retrieve_scored_rows(query_input, candidate_k=candidate_k, top_k=10)

    if len(scored_rows) == 0:
        return {
            "query_raw"          : raw_query,
            "query_processed"    : query_input,
            "final_department"   : None,
            "confidence_score"   : 0.0,
            "needs_manual_review": True,
            "scored_rows"        : scored_rows,
        }

    top1       = scored_rows.iloc[0]
    final_dept = top1["HNG_BR_NM"]
    top1_score = float(top1["score"])

    if len(scored_rows) > 1:
        gap        = top1_score - float(scored_rows.iloc[1]["score"])
        confidence = float(np.clip(top1_score * 0.7 + gap * 0.3, 0.0, 1.0))
    else:
        confidence = float(np.clip(top1_score, 0.0, 1.0))

    confidence   = round(confidence, 4)
    needs_review = confidence < 0.4

    return {
        "query_raw"          : raw_query,
        "query_processed"    : query_input,
        "final_department"   : final_dept,
        "confidence_score"   : confidence,
        "needs_manual_review": needs_review,
        "scored_rows"        : scored_rows,
    }


def print_prediction(result):
    print("=" * 60)
    print("[전처리 후 query]")
    print(result["query_processed"])
    print(f"\n[참조 행 top-10]")
    for _, row in result["scored_rows"].head(10).iterrows():
        print(f"  {int(row['rank'])}. {row['HNG_BR_NM']} | score={row['score']:.4f}")
    print(f"\n[최종] {result['final_department']}")
    conf  = result["confidence_score"]
    level = "높음" if conf >= 0.7 else ("중간" if conf >= 0.4 else "낮음 — 수동검토 필요")
    print(f"[신뢰도] {conf:.4f}  ({level})")
    print("=" * 60)


In [ ]:
# ============================================================
# 10. 단건 테스트
# ============================================================
for fname in ["query1.txt", "query2.txt", "query3.txt"]:
    fpath = PROJECT_DIR / fname
    if not fpath.exists():
        print(f"{fname} 없음, 건너뜀")
        continue

    query_text = fpath.read_text(encoding="utf-8")
    result = predict_department(
        query=query_text,
        candidate_k=CANDIDATE_K,
        preprocess=True,
    )
    print_prediction(result)
    print()


In [ ]:
# ============================================================
# 11. 배치 예측 (필요 시 주석 해제)
# ============================================================
# def batch_predict(queries, top_n_departments=5):
#     rows = []
#     for q in tqdm(queries, desc="predict"):
#         r = predict_department(q, top_n_departments=top_n_departments)
#         row = {"query": q, "final_department": r["final_department"],
#                "confidence_score": r["confidence_score"]}
#         for i, dept_row in r["top5_departments"].iterrows():
#             row[f"top{i+1}"] = dept_row["HNG_BR_NM"]
#         rows.append(row)
#     return pd.DataFrame(rows)
#
# sample_queries = [
#     "신탁형 ISA 상품 구성과 운용 관련해서 상담 가능한 부서를 알려주세요.",
#     "투자상품 원금손실 민원이 있어 판매 과정 적정성 검토 담당 부서를 찾고 있습니다.",
# ]
# display(batch_predict(sample_queries))


In [ ]:
# ============================================================
# 12. 평가 데이터셋 정확도 계산 (필요 시 주석 해제)
# ============================================================
# 평가 파일 컬럼: "문장", "유관부서"
#
# TEST_XLSX_PATH = PROJECT_DIR / "test_set.xlsx"
#
# def evaluate(test_path):
#     if not Path(test_path).exists():
#         print(f"평가 파일 없음: {test_path}"); return
#     test_df = pd.read_excel(test_path)
#     preds = []
#     for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
#         r = predict_department(str(row["문장"]))
#         preds.append({
#             "final_department": r["final_department"],
#             "confidence_score": r["confidence_score"],
#         })
#     pred_df = pd.DataFrame(preds)
#     out = pd.concat([test_df.reset_index(drop=True), pred_df], axis=1)
#     out["correct_top1"] = (out["final_department"].astype(str)
#                            == out["유관부서"].astype(str))
#     print("Top-1 accuracy:", out["correct_top1"].mean())
#     return out
#
# evaluate(TEST_XLSX_PATH)


## 실행 순서 요약

1. **Colab GPU 설정** — 런타임 > 런타임 유형 변경 > GPU (A100/V100 권장, T4 가능)
   (추론만 할 경우 CPU 가능 — `TRAIN_MODE = False`)
2. **파일 업로드** — `업무담당규정.docx`, `dept.xlsx`를 `/content` 또는 Drive에 업로드
3. **Cell 4** — `TRAIN_MODE`, 배치 크기 확인 후 순서대로 실행
4. **GPU 메모리 부족 시** — `STAGE1_BATCH_SIZE` / `STAGE2_BATCH_SIZE` → 4로 감소
5. **재사용 시** (모델 이미 있으면):
   ```python
   TRAIN_MODE = False
   RUN_STAGE1_DOMAIN_TRAIN = False
   RUN_STAGE2_DEPT_TRAIN   = False
   ```

## v4.1 핵심 변경 요약 (v4 → v4.1)

| 항목 | v4 | v5 |
|------|----|----|
| base 모델 | `jhgan/ko-sroberta-multitask` (111M) | **`BAAI/bge-m3`** (570M) |
| max_seq_length | 256 tokens (하드코딩) | **8192 tokens** (모델 기본값) |
| 입력 쿼리 길이 제한 | 300자 | **제한 없음** |
| 학습 API | `model.fit()` (deprecated) | **`SentenceTransformerTrainer`** |
| 출력 디렉터리 | `stage2_dept_retriever/` | **`bge_m3_stage2/`** |
| Stage1 배치 | 32 | 8 (대형 모델 OOM 방지) |
| Stage2 배치 | 32 | 8 |
| LR | 2e-5 | **1e-5** (대형 모델 안정 학습) |
| Stage2 epochs | 1 | **2** (데이터 적음) |
| Augmentation query 길이 | 400자 | 600자 |
